In [1]:
!pip install rtdl_num_embeddings -q --no-index --find-links=./input/jane-street-import/rtdl_num_embeddings

In [2]:
import os, sys, gc
import pickle
import dill
import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from pytorch_lightning import LightningModule
import torch.optim
from tabm_reference import Model, make_parameter_groups
import warnings
import joblib
import gc
import warnings

warnings.filterwarnings("ignore")
pd.options.display.max_columns = None
sys.path.append("./input/jane-street-real-time-market-data-forecasting")

In [3]:
class CONFIG:
    seed = 42
    target_col = "responder_6"
    feature_cols = [f"feature_{idx:02d}" for idx in range(79)] + [
        f"responder_{idx}_lag_1" for idx in range(9)
    ]
    model_paths = [
        "./input/js-xs-nn-trained-model",
        "./input/js-with-lags-trained-xgb/result.pkl",
        "./input/als-e-106-pp0-40-xgb-5fold/result0.pkl",
        "./input/als-e-106-pp0-40-xgb-5fold/result1.pkl",
        "./input/als-e-106-pp0-40-xgb-5fold/result2.pkl",
        "./input/als-e-106-pp0-40-xgb-5fold/result3.pkl",
        "./input/als-e-106-pp0-40-xgb-5fold/result4.pkl",
    ]

valid = (
    pl.scan_parquet(
        f"./input/js24-preprocessing-create-lags/validation.parquet/"
    )
    .collect()
    .to_pandas()
)

xgb_model = None
with open(CONFIG.model_paths[2], "rb") as fp:
    result = pickle.load(fp)
    xgb_model = result["model"]
xgb_feature_cols = ["symbol_id", "time_id"] + CONFIG.feature_cols

xgb_model2 = None
with open(CONFIG.model_paths[4], "rb") as fp:
    result = pickle.load(fp)
    xgb_model2 = result["model"]

xgb_model4 = None
with open(CONFIG.model_paths[6], "rb") as fp:
    result = pickle.load(fp)
    xgb_model4 = result["model"]

xgb_model5 = None
with open(CONFIG.model_paths[1], "rb") as fp:
    result = pickle.load(fp)
    xgb_model5 = result["model"]

def r2_val(y_true, y_pred, sample_weight):
    r2 = 1 - np.average((y_pred - y_true) ** 2, weights=sample_weight) / (
        np.average((y_true) ** 2, weights=sample_weight) + 1e-38
    )
    return r2

class NN(LightningModule):
    def __init__(self, input_dim, hidden_dims, dropouts, lr, weight_decay):
        super().__init__()
        self.save_hyperparameters()
        layers = []
        in_dim = input_dim
        for i, hidden_dim in enumerate(hidden_dims):
            layers.append(nn.BatchNorm1d(in_dim))
            if i > 0:
                layers.append(nn.SiLU())
            if i < len(dropouts):
                layers.append(nn.Dropout(dropouts[i]))
            layers.append(nn.Linear(in_dim, hidden_dim))
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, 1))
        layers.append(nn.Tanh())
        self.model = nn.Sequential(*layers)
        self.lr = lr
        self.weight_decay = weight_decay
        self.validation_step_outputs = []

    def forward(self, x):
        return 5 * self.model(x).squeeze(-1)

    def training_step(self, batch):
        x, y, w = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y, reduction="none") * w
        loss = loss.mean()
        self.log(
            "train_loss",
            loss,
            on_step=False,
            on_epoch=True,
            batch_size=x.size(0),
        )
        return loss

    def validation_step(self, batch):
        x, y, w = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y, reduction="none") * w
        loss = loss.mean()
        self.log(
            "val_loss",
            loss,
            on_step=False,
            on_epoch=True,
            batch_size=x.size(0),
        )
        self.validation_step_outputs.append((y_hat, y, w))
        return loss

    def on_validation_epoch_end(self):
        if not self.trainer.sanity_checking:
            y = (
                torch.cat([x[1] for x in self.validation_step_outputs])
                .cpu()
                .numpy()
            )
            prob = (
                torch.cat([x[0] for x in self.validation_step_outputs])
                .cpu()
                .numpy()
            )
            weights = (
                torch.cat([x[2] for x in self.validation_step_outputs])
                .cpu()
                .numpy()
            )
            val_r_square = r2_val(y, prob, weights)
            self.log(
                "val_r_square",
                val_r_square,
                prog_bar=True,
                on_step=False,
                on_epoch=True,
            )
        self.validation_step_outputs.clear()

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(
            self.parameters(), lr=self.lr, weight_decay=self.weight_decay
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=5, verbose=True
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
            },
        }

    def on_train_epoch_end(self):
        if not self.trainer.sanity_checking:
            epoch = self.trainer.current_epoch
            metrics = {
                k: v.item() if isinstance(v, torch.Tensor) else v
                for k, v in self.trainer.logged_metrics.items()
            }
            formatted_metrics = {k: f"{v:.5f}" for k, v in metrics.items()}
            print(f"Epoch {epoch}: {formatted_metrics}")

N_folds = 5
models = []
for fold in range(N_folds):
    checkpoint_path = f"{CONFIG.model_paths[0]}/nn_{fold}.model"
    model = NN.load_from_checkpoint(checkpoint_path)
    models.append(model)

del valid
gc.collect()

367

In [4]:
lags_: pl.DataFrame | None = None

def predict_nn_xgb(
    test: pl.DataFrame, lags: pl.DataFrame | None
) -> pl.DataFrame | pd.DataFrame:
    global lags_
    if lags is not None:
        lags_ = lags
    predictions_nn = test.select(
        "row_id",
        pl.lit(0.0).alias(
            "responder_6",
        ),
    )
    lags = (
        lags_.clone()
        .group_by(["date_id", "symbol_id"], maintain_order=True)
        .last()
    )
    test = test.join(lags, on=["date_id", "symbol_id"], how="left")
    preds_xgb = np.zeros((test.shape[0],))
    preds_nn = np.zeros((test.shape[0],))
    preds_xgb += xgb_model.predict(test[xgb_feature_cols].to_pandas()) * 0.25
    preds_xgb += xgb_model2.predict(test[xgb_feature_cols].to_pandas()) * 0.25
    preds_xgb += xgb_model4.predict(test[xgb_feature_cols].to_pandas()) * 0.25
    preds_xgb += xgb_model5.predict(test[xgb_feature_cols].to_pandas()) * 0.25
    test_input = test[CONFIG.feature_cols].to_pandas()
    test_input = test_input.fillna(method="ffill").fillna(0)
    test_input = torch.FloatTensor(test_input.values)
    with torch.no_grad():
        for i, nn_model in enumerate(models):
            nn_model.eval()
            preds_nn += nn_model(test_input).cpu().numpy() / len(models)
    preds = 0.55 * preds_xgb + 0.45 * preds_nn
    predictions_nn = test.select("row_id").with_columns(
        pl.Series(
            name="responder_6",
            values=np.clip(preds, a_min=-5, a_max=5),
            dtype=pl.Float64,
        )
    )
    return predictions_nn

In [5]:
feature_list = [f"feature_{idx:02d}" for idx in range(79) if idx != 61]
target_col = "responder_6"
feature_test = feature_list + [f"responder_{idx}_lag_1" for idx in range(9)]
feature_cat = ["feature_09", "feature_10", "feature_11"]
feature_cont = [item for item in feature_test if item not in feature_cat]
batch_size = 8192
std_feature = [i for i in feature_list if i not in feature_cat] + [
    f"responder_{idx}_lag_1" for idx in range(9)
]
data_stats = joblib.load("./input/my-own-js/data_stats.pkl")
means = data_stats["mean"]
stds = data_stats["std"]

def standardize(df, feature_cols, means, stds):
    return df.with_columns(
        [
            ((pl.col(col) - means[col]) / stds[col]).alias(col)
            for col in feature_cols
        ]
    )

category_mappings = {'feature_09': {2: 0, 4: 1, 9: 2, 11: 3, 12: 4, 14: 5, 15: 6, 25: 7, 26: 8, 30: 9, 34: 10, 42: 11, 44: 12, 46: 13, 49: 14, 50: 15, 57: 16, 64: 17, 68: 18, 70: 19, 81: 20, 82: 21},
 'feature_10': {1: 0, 2: 1, 3: 2, 4: 3, 5: 4, 6: 5, 7: 6, 10: 7, 12: 8},
 'feature_11': {9: 0, 11: 1, 13: 2, 16: 3, 24: 4, 25: 5, 34: 6, 40: 7, 48: 8, 50: 9, 59: 10, 62: 11, 63: 12, 66: 13,
  76: 14, 150: 15, 158: 16, 159: 17, 171: 18, 195: 19, 214: 20, 230: 21, 261: 22, 297: 23, 336: 24, 376: 25, 388: 26, 410: 27, 522: 28, 534: 29, 539: 30},
 'symbol_id': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11, 12: 12, 13: 13, 14: 14, 15: 15, 16: 16, 17: 17, 18: 18, 19: 19,
  20: 20, 21: 21, 22: 22, 23: 23, 24: 24, 25: 25, 26: 26, 27: 27, 28: 28, 29: 29, 30: 30, 31: 31, 32: 32, 33: 33, 34: 34, 35: 35, 36: 36, 37: 37, 38: 38},
 'time_id' : {i : i for i in range(968)}}

def encode_column(df, column, mapping):
    max_value = max(mapping.values())

    def encode_category(category):
        return mapping.get(category, max_value + 1)

    return df.with_columns(
        pl.col(column).map_elements(encode_category).alias(column)
    )

class R2Loss(nn.Module):
    def __init__(self):
        super(R2Loss, self).__init__()

    def forward(self, y_pred, y_true):
        mse_loss = torch.sum((y_pred - y_true) ** 2)
        var_y = torch.sum(y_true**2)
        loss = mse_loss / (var_y + 1e-38)
        return loss

class NN(LightningModule):
    def __init__(
        self, n_cont_features, cat_cardinalities, n_classes, lr, weight_decay
    ):
        super().__init__()
        self.save_hyperparameters()
        self.k = 16
        self.model = Model(
            n_num_features=n_cont_features,
            cat_cardinalities=cat_cardinalities,
            n_classes=n_classes,
            backbone={
                "type": "MLP",
                "n_blocks": 3,
                "d_block": 512,
                "dropout": 0.25,
            },
            bins=None,
            num_embeddings=None,
            arch_type="tabm",
            k=self.k,
        )
        self.lr = lr
        self.weight_decay = weight_decay
        self.training_step_outputs = []
        self.validation_step_outputs = []
        self.loss_fn = R2Loss()

    def forward(self, x_cont, x_cat):
        return self.model(x_cont, x_cat).squeeze(-1)

    def training_step(self, batch):
        x_cont, x_cat, y, w, w_y = batch
        x_cont = x_cont + torch.randn_like(x_cont) * 0.02
        y_hat = self(x_cont, x_cat)
        loss = self.loss_fn(y_hat.flatten(0, 1), y.repeat_interleave(self.k))
        self.log(
            "train_loss",
            loss,
            on_step=True,
            on_epoch=True,
            prog_bar=True,
            logger=True,
            batch_size=x_cont.size(0),
        )
        self.training_step_outputs.append((y_hat.mean(1), y, w))
        return loss

    def validation_step(self, batch):
        x_cont, x_cat, y, w, w_y = batch
        x_cont = x_cont + torch.randn_like(x_cont) * 0.02
        y_hat = self(x_cont, x_cat)
        loss = self.loss_fn(y_hat.flatten(0, 1), y.repeat_interleave(self.k))
        self.log(
            "val_loss",
            loss,
            on_step=False,
            on_epoch=True,
            prog_bar=True,
            logger=True,
            batch_size=x_cont.size(0),
        )
        self.validation_step_outputs.append((y_hat.mean(1), y, w))
        return loss

    def on_validation_epoch_end(self):
        y = (
            torch.cat([x[1] for x in self.validation_step_outputs])
            .cpu()
            .numpy()
        )
        if self.trainer.sanity_checking:
            prob = (
                torch.cat([x[0] for x in self.validation_step_outputs])
                .cpu()
                .numpy()
            )
        else:
            prob = (
                torch.cat([x[0] for x in self.validation_step_outputs])
                .cpu()
                .numpy()
            )
            weights = (
                torch.cat([x[2] for x in self.validation_step_outputs])
                .cpu()
                .numpy()
            )
            val_r_square = r2_val(y, prob, weights)
            self.log(
                "val_r_square",
                val_r_square,
                prog_bar=True,
                on_step=False,
                on_epoch=True,
            )
        self.validation_step_outputs.clear()

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            make_parameter_groups(self.model),
            lr=self.lr,
            weight_decay=self.weight_decay,
        )
        return {"optimizer": optimizer}

    def on_train_epoch_end(self):
        if self.trainer.sanity_checking:
            return
        y = torch.cat([x[1] for x in self.training_step_outputs]).cpu().numpy()
        prob = (
            torch.cat([x[0] for x in self.training_step_outputs])
            .detach()
            .cpu()
            .numpy()
        )
        weights = (
            torch.cat([x[2] for x in self.training_step_outputs]).cpu().numpy()
        )
        train_r_square = r2_val(y, prob, weights)
        self.log(
            "train_r_square",
            train_r_square,
            prog_bar=True,
            on_step=False,
            on_epoch=True,
        )
        self.training_step_outputs.clear()
        epoch = self.trainer.current_epoch
        metrics = {
            k: v.item() if isinstance(v, torch.Tensor) else v
            for k, v in self.trainer.logged_metrics.items()
        }
        formatted_metrics = {k: f"{v:.5f}" for k, v in metrics.items()}
        print(f"Epoch {epoch}: {formatted_metrics}")

class custom_args:
    def __init__(self):
        self.usegpu = True
        self.gpuid = 0
        self.seed = 42
        self.model = "nn"
        self.use_wandb = False
        self.project = "js-tabm-with-lags"
        self.dname = "./input_df/"
        self.loader_workers = 10
        self.bs = 8192
        self.lr = 1e-3
        self.weight_decay = 8e-4
        self.n_cont_features = 84
        self.n_cat_features = 5
        self.n_classes = None
        self.cat_cardinalities = [23, 10, 32, 40, 969]
        self.patience = 7
        self.max_epochs = 10
        self.N_fold = 5

my_args = custom_args()
device = "cpu"
model = NN.load_from_checkpoint("./input/my-own-js/tabm_epochepoch03.ckpt").to(
    device
)

In [6]:
lags_: pl.DataFrame | None = None
lags_history = None

def predict_tabm(
    test: pl.DataFrame, lags: pl.DataFrame | None
) -> pl.DataFrame | pd.DataFrame:
    global lags_, lags_history
    if lags is not None:
        lags_ = lags
    for col in feature_cat + ["symbol_id", "time_id"]:
        test = encode_column(test, col, category_mappings[col])
    predictions = test.select("row_id", pl.lit(0.0).alias("responder_6"))
    symbol_ids = test.select("symbol_id").to_numpy()[:, 0]
    time_id = test.select("time_id").to_numpy()[0]
    timie_id_array = test.select("time_id").to_numpy()[:, 0]
    if time_id == 0:
        lags = lags.with_columns(pl.col("time_id").cast(pl.Int64))
        lags = lags.with_columns(pl.col("symbol_id").cast(pl.Int64))
        lags_history = lags
        lags = lags.filter(pl.col("time_id") == 0)
        test = test.join(lags, on=["time_id", "symbol_id"], how="left")
    else:
        lags = lags_history.filter(pl.col("time_id") == time_id)
        test = test.join(lags, on=["time_id", "symbol_id"], how="left")
    test = test.with_columns(
        [
            pl.col(col).fill_null(0)
            for col in feature_list
            + [f"responder_{idx}_lag_1" for idx in range(9)]
        ]
    )
    test = standardize(test, std_feature, means, stds)
    X_test = test[feature_test].to_numpy()
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    symbol_tensor = torch.tensor(symbol_ids, dtype=torch.float32).to(device)
    time_tensor = torch.tensor(timie_id_array, dtype=torch.float32).to(device)
    X_cat = X_test_tensor[:, [9, 10, 11]]
    X_cont = X_test_tensor[
        :, [i for i in range(X_test_tensor.shape[1]) if i not in [9, 10, 11]]
    ]
    X_cat = (
        torch.concat(
            [X_cat, symbol_tensor.unsqueeze(-1), time_tensor.unsqueeze(-1)],
            axis=1,
        )
    ).to(torch.int64)
    model.eval()
    with torch.no_grad():
        outputs = model(X_cont, X_cat)
        preds = outputs.squeeze(-1).cpu().numpy()
        preds = preds.mean(1)
    predictions = test.select("row_id").with_columns(
        pl.Series(
            name="responder_6",
            values=np.clip(preds, a_min=-5, a_max=5),
            dtype=pl.Float64,
        )
    )
    return predictions

In [7]:
def load_from_dill(model_name, model_path=None, file_ext=".dill"):
    model_object = None
    with open(f"{model_path}/{model_name}{file_ext}", "rb") as file_handle:
        model_object = dill.load(file_handle)
    return model_object

rdg = load_from_dill(model_name="Ridge", model_path="./input/jsridgev01011635")

def predict_ridge(test, lags):
    cols = [f"feature_{i:02}" for i in range(79)]
    predictions = test.select(
        "row_id",
        pl.lit(0.0).alias("responder_6"),
    )
    test_preds = rdg.predict(test[cols].to_pandas().fillna(3).values)
    predictions = predictions.with_columns(
        pl.Series("responder_6", test_preds.ravel())
    )
    return predictions

In [8]:
def predict(
    test: pl.DataFrame, lags: pl.DataFrame | None
) -> pl.DataFrame | pd.DataFrame:
    pd_nn_xgb = predict_nn_xgb(test, lags).to_pandas()
    pd_ridge = predict_ridge(test, lags).to_pandas()
    pd_tabm = predict_tabm(test, lags).to_pandas()
    pd_nn_xgb = pd_nn_xgb.rename(columns={"responder_6": "col_nn_xgb"})
    pd_ridge = pd_ridge.rename(columns={"responder_6": "col_ridge"})
    pd_tabm = pd_tabm.rename(columns={"responder_6": "col_tabm"})
    pds = pd.merge(pd_nn_xgb, pd_ridge, on=["row_id"])
    pds = pd.merge(pds, pd_tabm, on=["row_id"])

    # Weights from ensemble-tuning.ipynb using this predict function
    e_weights = [0.8099994, -0.2236468, 0.81511762, 0.0037626]
    pds["responder_6"] = (
        pds["col_nn_xgb"] * e_weights[0]
        + pds["col_ridge"] * e_weights[1]
        + pds["col_tabm"] * e_weights[2]
        + e_weights[3]
    )
    predictions = test.select("row_id", pl.lit(0.0).alias("responder_6"))
    pred = pds["responder_6"].to_numpy()
    predictions = predictions.with_columns(
        pl.Series("responder_6", pred.ravel())
    )
    return predictions

In [9]:
test = pl.read_parquet(
    "./input/jane-street-real-time-market-data-forecasting/test.parquet"
)
lags = pl.read_parquet(
    "./input/jane-street-real-time-market-data-forecasting/lags.parquet"
)
predict(test, lags)

row_id,responder_6
i64,f64
0,-0.051925
1,-0.05044
2,-0.035298
3,-0.024921
4,-0.083517
…,…
34,-0.016333
35,-0.03256
36,-0.025209
